In [ ]:
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torch.nn.functional as F
from math import log10
from skimage.metrics import structural_similarity as ssim

class LensingSRDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, transform=None):
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.transform = transform
        self.filenames = sorted([f for f in os.listdir(hr_dir) if f.endswith('.npy')])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        hr_img = np.load(os.path.join(self.hr_dir, fname))
        lr_img = np.load(os.path.join(self.lr_dir, fname))
        
        hr_tensor = torch.from_numpy(hr_img).float()
        lr_tensor = torch.from_numpy(lr_img).float()
        
        
        if hr_tensor.ndim == 2: hr_tensor = hr_tensor.unsqueeze(0)
        if lr_tensor.ndim == 2: lr_tensor = lr_tensor.unsqueeze(0)
        
        if hr_tensor.shape[0] > 1 and hr_tensor.ndim == 3: hr_tensor = hr_tensor[0:1, :, :]
        if lr_tensor.shape[0] > 1 and lr_tensor.ndim == 3: lr_tensor = lr_tensor[0:1, :, :]

        if self.transform:
            seed = np.random.randint(2147483647)
            torch.manual_seed(seed)
            hr_tensor = self.transform(hr_tensor)
            torch.manual_seed(seed)
            lr_tensor = self.transform(lr_tensor)
            
        return lr_tensor, hr_tensor

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
])

lr_path = '/kaggle/input/datasets/bryanbradfo/gsoc-deeplense-image-super-resolution/Dataset/LR'
hr_path = '/kaggle/input/datasets/bryanbradfo/gsoc-deeplense-image-super-resolution/Dataset/HR'

train_ds = LensingSRDataset(lr_path, hr_path, transform=train_tf)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

In [5]:
from timm.layers import DropPath, to_2tuple, trunc_normal_

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))

class SwinIR(nn.Module):
    def __init__(self, in_chans=1, embed_dim=96, upscale=2):
        super(SwinIR, self).__init__()
        self.conv_first = nn.Conv2d(in_chans, embed_dim, 3, 1, 1)
        self.upsample = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim * (upscale ** 2), 3, 1, 1),
            nn.PixelShuffle(upscale),
            nn.Conv2d(embed_dim, in_chans, 3, 1, 1)
        )
    def forward(self, x):
        return self.upsample(self.conv_first(x))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SwinIR(upscale=2).to(device)
optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = nn.L1Loss()
scaler = torch.amp.GradScaler('cuda')

for epoch in range(20):
    model.train()
    epoch_loss = 0
    for lr, hr in train_loader:
        lr, hr = lr.to(device), hr.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            sr = model(lr)

            if sr.shape != hr.shape:
                sr = F.interpolate(sr, size=(hr.shape[2], hr.shape[3]), mode='bilinear')
            loss = criterion(sr, hr)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(train_loader):.6f}")

Epoch 1 Loss: 0.014551
Epoch 2 Loss: 0.008676
Epoch 3 Loss: 0.006952
Epoch 4 Loss: 0.005669
Epoch 5 Loss: 0.005338
Epoch 6 Loss: 0.005156
Epoch 7 Loss: 0.005102
Epoch 8 Loss: 0.005077
Epoch 9 Loss: 0.005039
Epoch 10 Loss: 0.005002
Epoch 11 Loss: 0.004984
Epoch 12 Loss: 0.004965
Epoch 13 Loss: 0.004962
Epoch 14 Loss: 0.004960
Epoch 15 Loss: 0.004959
Epoch 16 Loss: 0.004959
Epoch 17 Loss: 0.004958
Epoch 18 Loss: 0.004958
Epoch 19 Loss: 0.004958
Epoch 20 Loss: 0.004958


In [7]:
def evaluate(model, loader):
    model.eval()
    mse_sum, psnr_sum, ssim_sum = 0, 0, 0
    with torch.no_grad():
        for lr, hr in loader:
            lr, hr = lr.to(device), hr.to(device)
            sr = model(lr).clamp(0, 1)
            if sr.shape != hr.shape:
                sr = F.interpolate(sr, size=(hr.shape[2], hr.shape[3]), mode='bilinear')
            
            mse = F.mse_loss(sr, hr).item()
            mse_sum += mse
            psnr_sum += 10 * log10(1 / (mse + 1e-10))
            
            sr_img = sr[0].cpu().numpy().squeeze()
            hr_img = hr[0].cpu().numpy().squeeze()
            ssim_sum += ssim(sr_img, hr_img, data_range=1)
            
    n = len(loader)
    print(f"Final Metrics -> MSE: {mse_sum/n:.6f}, PSNR: {psnr_sum/n:.2f}dB, SSIM: {ssim_sum/n:.4f}")

evaluate(model, train_loader)
torch.save(model.state_dict(), 'lensing_sr_final.pth')

Final Metrics -> MSE: 0.000061, PSNR: 42.13dB, SSIM: 0.9774
